# G22: the G21 on-policy-distilled E4B adapter with its reasoning capped at 512 tokens, plus a base-E4B control at the same cap

No training. Does the distilled adapter carry any skill beyond its longer reasoning? Wiring check -> load + package the G21 adapter -> vLLM + capture proxy -> adapter-effect + smoke gates (incl. the capped budget in real requests) -> 75-task round with the adapter -> 75-task control round with the base model.

In [ ]:
import glob, importlib, os, shutil, subprocess, sys
from pathlib import Path
os.environ['LITELLM_LOCAL_MODEL_COST_MAP'] = 'True'
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['VLLM_WORKER_MULTIPROC_METHOD'] = 'spawn'
os.environ['VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS'] = '1'
os.environ['VLLM_ENGINE_READY_TIMEOUT_S'] = '1200'
os.environ['VLLM_NO_USAGE_STATS'] = '1'
os.environ['OTEL_SDK_DISABLED'] = 'true'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
WHEELHOUSE_DIR = Path('/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse')
for pat in ('/usr/local/lib/python*/dist-packages/*cutlass*.pth', '/usr/local/lib/python*/site-packages/*cutlass*.pth'):
    for pth in glob.glob(pat):
        try: os.unlink(pth)
        except OSError: pass
tmp = Path('/tmp/wheelhouse'); tmp.mkdir(parents=True, exist_ok=True)
for w in WHEELHOUSE_DIR.glob('*.whl'):
    if 'cutlass' in w.name.lower(): continue
    name = w.name.replace('cu128', '+cu128') if ('cu128' in w.name and '+' not in w.name) else w.name
    t = tmp / name
    if not t.exists(): os.symlink(w, t)
wheels = sorted(str(w) for w in tmp.glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--force-reinstall', *wheels], check=True)
importlib.invalidate_caches()
print('installed', len(wheels), 'wheels')

## Setup

In [ ]:
import json, os, random, shutil, subprocess, sys, threading, time, traceback
from pathlib import Path
WORK = Path('/kaggle/working'); G3 = WORK / 'g22'; CODE = G3 / 'code'; CODE.mkdir(parents=True, exist_ok=True); WORKING_DIR = WORK
R = {'started': time.strftime('%Y-%m-%d %H:%M:%S')}
def save_results(): (G3 / 'g22_summary.json').write_text(json.dumps(R, indent=1, default=str))
BUDGET_MIN = 270; T_START = time.time()
def _watchdog():
    R['watchdog'] = f'fired after {BUDGET_MIN} min'; save_results(); os._exit(3)
_wd = threading.Timer(BUDGET_MIN * 60, _watchdog); _wd.daemon = True; _wd.start()
for n, txt in {"train_g3.py": "\"\"\"G3: LoRA SFT of Gemma 4 31B (bf16 QAT-unquantized checkpoint, 4 x L4) on token-exact sequences captured in G2.\n\nInput jsonl rows: {\"input_ids\": [...], \"loss_spans\": [[start, end), ...], \"n_loss_tokens\": int, \"task\": str, ...} (scripts/g2/build_sft_dataset.py).\nThe loss is the mean negative log-likelihood over the tokens inside the spans (token k is predicted from position k-1). Gradient accumulation groups\nare normalised by their total number of loss tokens, so every loss token has equal weight.\nMemory: sequence-chunked decoder layers (scripts/g1/gemma4_memfix.py, exact, verified at the real 31B dimensions) + chunked lm_head loss: trained and probed\non the L4x4 up to 18,432 tokens in G1d. The environment fixes found in G1 are kept: torchao hidden from peft, checkpoint staged to local disk by the notebook.\n\nEnvironment variables: G3_MODEL, G3_DATA, G3_OUT (required); G3_TINY=1 (tiny random model for local tests); G3_ACC (default 2); G3_LR (2e-4);\nG3_EPOCHS (1); G3_HF_GC (1 = HF gradient checkpointing, for E4B/E2B); G3_TINY_PLE (1 = tiny test model with per-layer inputs); G3_WARMUP (3 optimizer steps); G3_SEED (0); G3_MAXMEM (JSON per-GPU limits); G3_MAX_SEQS (use only the first N sequences, tests); G3_CKPT_EVERY (10).\n\"\"\"\n\nimport contextlib\nimport gc\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\n\nos.environ.setdefault(\"PYTORCH_CUDA_ALLOC_CONF\", \"expandable_segments:True\")\nsys.modules[\"torchao\"] = None   # Kaggle's torchao 0.10.0 is rejected by peft 0.19.1 and we do not use it\nimport torch\nimport torch.nn.functional as F\nimport transformers\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent / \"g1\"))\nimport gemma4_memfix as MF  # noqa: E402\n\nMODEL, DATA, OUT = os.environ[\"G3_MODEL\"], os.environ[\"G3_DATA\"], Path(os.environ[\"G3_OUT\"])\nTINY = os.environ.get(\"G3_TINY\") == \"1\"\nACC = int(os.environ.get(\"G3_ACC\", \"2\"))\nLR = float(os.environ.get(\"G3_LR\", \"2e-4\"))\nEPOCHS = int(os.environ.get(\"G3_EPOCHS\", \"1\"))\nWARMUP = int(os.environ.get(\"G3_WARMUP\", \"3\"))\nSEED = int(os.environ.get(\"G3_SEED\", \"0\"))\nMAXMEM = json.loads(os.environ.get(\"G3_MAXMEM\", \"null\"))\nMAX_SEQS = int(os.environ.get(\"G3_MAX_SEQS\", \"0\"))\nCKPT_EVERY = int(os.environ.get(\"G3_CKPT_EVERY\", \"10\"))\nTEST_MEMFRAC = os.environ.get(\"G3_TEST_MEMFRAC\")\nHF_GC = os.environ.get(\"G3_HF_GC\") == \"1\"       # HF gradient checkpointing: needed for models whose layers fall back to the stock forward (E4B/E2B: per-layer inputs, shared KV)\nTINY_PLE = os.environ.get(\"G3_TINY_PLE\") == \"1\"  # tiny test model WITH per-layer inputs and shared-KV layers (E4B-like)\nTARGET_RE = r\".*language_model\\.layers\\.\\d+\\.(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|mlp\\.(gate_proj|up_proj|down_proj))\"\nOUT.mkdir(parents=True, exist_ok=True)\nres: dict = {\"stage\": \"start\", \"tiny\": TINY, \"acc\": ACC, \"lr\": LR, \"epochs\": EPOCHS, \"warmup\": WARMUP, \"seed\": SEED,\n             \"torch\": torch.__version__, \"transformers\": transformers.__version__, \"n_gpus\": torch.cuda.device_count()}\n\n\ndef save() -> None:\n    (OUT / \"train_metrics.json\").write_text(json.dumps(res, indent=1), encoding=\"utf-8\")\n\n\ndef log(*a) -> None:\n    print(\"[g3]\", *a, flush=True)\n\n\ndef gpu_peaks() -> list[float]:\n    return [round(torch.cuda.max_memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())]\n\n\ndef load_model():\n    cls = transformers.Gemma4ForConditionalGeneration\n    MF.install()\n    MF.install_layers(chunk_sliding=1024, chunk_global=128)\n    if TINY:\n        from transformers import Gemma4Config, Gemma4TextConfig\n        t = Gemma4TextConfig(vocab_size=1000, hidden_size=64, intermediate_size=128, num_hidden_layers=6, num_attention_heads=4, num_key_value_heads=2,\n                             head_dim=16, sliding_window=32, layer_types=([\"sliding_attention\", \"sliding_attention\", \"full_attention\", \"sliding_attention\", \"sliding_attention\", \"full_attention\"] if TINY_PLE\n                                     else [\"sliding_attention\"] * 5 + [\"full_attention\"]), final_logit_softcapping=30.0,\n                             hidden_size_per_layer_input=(16 if TINY_PLE else 0), vocab_size_per_layer_input=1000, num_kv_shared_layers=(2 if TINY_PLE else 0),\n                             use_double_wide_mlp=False, max_position_embeddings=65536)\n        cfg = Gemma4Config(text_config=t.to_dict(), vision_config=dict(hidden_size=32, intermediate_size=64, num_hidden_layers=1, num_attention_heads=2,\n                           num_key_value_heads=2, head_dim=16, global_head_dim=16), audio_config=None)\n        return cls._from_config(cfg, attn_implementation=\"sdpa\").to(torch.bfloat16).to(\"cuda:0\" if torch.cuda.is_available() else \"cpu\")\n    kw = dict(dtype=torch.bfloat16, device_map=\"auto\", attn_implementation=\"sdpa\")\n    if MAXMEM:\n        kw[\"max_memory\"] = {int(k): v for k, v in MAXMEM.items()}\n    return cls.from_pretrained(MODEL, **kw)\n\n\ndef main() -> None:\n    seqs = [json.loads(line) for line in open(DATA, encoding=\"utf-8\")]\n    if MAX_SEQS:\n        seqs = seqs[:MAX_SEQS]\n    res[\"data\"] = {\"sequences\": len(seqs), \"tokens\": sum(len(s[\"input_ids\"]) for s in seqs), \"loss_tokens\": sum(s[\"n_loss_tokens\"] for s in seqs),\n                   \"max_len\": max(len(s[\"input_ids\"]) for s in seqs), \"tasks\": len({s[\"task\"] for s in seqs})}\n    log(\"data\", res[\"data\"])\n    save()\n\n    t0 = time.perf_counter()\n    model = load_model()\n    res[\"load_seconds\"] = round(time.perf_counter() - t0, 1)\n    res[\"mem_after_load_gib\"] = [round(torch.cuda.memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())]\n    log(\"loaded in\", res[\"load_seconds\"], \"s; mem after load (GiB):\", res[\"mem_after_load_gib\"])\n    save()\n\n    if TEST_MEMFRAC and torch.cuda.is_available():\n        torch.cuda.set_per_process_memory_fraction(float(TEST_MEMFRAC))\n    from peft import LoraConfig, get_peft_model\n    model.config.use_cache = False\n    if HF_GC:\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\"use_reentrant\": False})\n        log(\"HF gradient checkpointing enabled\")\n    pm = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias=\"none\", target_modules=TARGET_RE, task_type=\"CAUSAL_LM\"))\n    params = [p for p in pm.parameters() if p.requires_grad]\n    res[\"trainable_params_m\"] = round(sum(p.numel() for p in params) / 1e6, 1)\n    assert any(n.endswith(\".lora_A.default\") for n, _ in pm.named_modules()), \"TARGET_RE matched no module\"\n    pm.train()\n    dev = model.get_input_embeddings().weight.device\n    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)\n    inner, lm_head = model.model, model.lm_head\n    softcap = getattr(model.config.get_text_config(), \"final_logit_softcapping\", None)\n    from torch.utils.checkpoint import checkpoint\n    CH = 512\n    base_alloc = [torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())]\n\n    def chunk_loss(h, tgt):\n        logits = lm_head(h).float()\n        if softcap:\n            logits = torch.tanh(logits / softcap) * softcap\n        return F.cross_entropy(logits, tgt, reduction=\"sum\")\n\n    def fwd_bwd(seq: dict, denom: float) -> tuple[float, int]:\n        ids = seq[\"input_ids\"]\n        pos = sorted({j - 1 for s, e in seq[\"loss_spans\"] for j in range(max(s, 1), min(e, len(ids)))})\n        x = torch.tensor([ids], device=dev)\n        hs = inner(input_ids=x, attention_mask=torch.ones_like(x), use_cache=False).last_hidden_state[0]\n        hd = lm_head.weight.device\n        h_sel = hs[torch.tensor(pos, device=hs.device)].to(hd)\n        tgt = torch.tensor([ids[p + 1] for p in pos], device=hd)\n        total = None\n        for a in range(0, len(pos), CH):\n            part = checkpoint(chunk_loss, h_sel[a:a + CH], tgt[a:a + CH], use_reentrant=False)\n            total = part if total is None else total + part\n        (total / denom).backward()\n        return float(total.detach()), len(pos)\n\n    def cleanup() -> None:\n        opt.zero_grad(set_to_none=True)\n        gc.collect()\n        torch.cuda.empty_cache()\n        gc.collect()\n        torch.cuda.empty_cache()\n\n    def lr_at(step: int, total: int) -> float:       # linear warmup, then cosine decay to 10% of the peak\n        if step < WARMUP:\n            return LR * (step + 1) / WARMUP\n        prog = (step - WARMUP) / max(1, total - WARMUP)\n        return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))\n\n    order = list(range(len(seqs)))\n    groups = []\n    for ep in range(EPOCHS):\n        random.Random(SEED + ep).shuffle(order)\n        groups += [order[i:i + ACC] for i in range(0, len(order), ACC)]\n    total_steps = len(groups)\n    res.update({\"stage\": \"training\", \"optimizer_steps_planned\": total_steps, \"steps\": [], \"oom_events\": []})\n    for i in range(torch.cuda.device_count()):\n        torch.cuda.reset_peak_memory_stats(i)\n    t_train0 = time.perf_counter()\n    tok_done = 0\n    for step, grp in enumerate(groups):\n        lr = lr_at(step, total_steps)\n        for g in opt.param_groups:\n            g[\"lr\"] = lr\n        denom = float(sum(seqs[i][\"n_loss_tokens\"] for i in grp))\n        loss_sum, n_loss, n_tok, t1 = 0.0, 0, 0, time.perf_counter()\n        failed = False\n        for i in grp:\n            oom = False\n            try:\n                ls, nl = fwd_bwd(seqs[i], denom)\n                loss_sum += ls\n                n_loss += nl\n                n_tok += len(seqs[i][\"input_ids\"])\n            except torch.cuda.OutOfMemoryError:\n                oom = True\n            if oom:     # handled outside the except block so the failed step's tensors can be freed\n                res[\"oom_events\"].append({\"step\": step, \"task\": seqs[i][\"task\"], \"tokens\": len(seqs[i][\"input_ids\"])})\n                log(\"OOM on\", seqs[i][\"task\"], len(seqs[i][\"input_ids\"]), \"tokens; skipping this group\")\n                failed = True\n                break\n        if failed:\n            cleanup()\n            save()\n            continue\n        gn = float(torch.nn.utils.clip_grad_norm_(params, 1.0))\n        opt.step()\n        opt.zero_grad(set_to_none=True)\n        torch.cuda.synchronize()\n        dt = time.perf_counter() - t1\n        tok_done += n_tok\n        rec = {\"step\": step, \"lr\": round(lr, 7), \"loss\": round(loss_sum / max(1, n_loss), 4), \"grad_norm\": round(gn, 4), \"tokens\": n_tok, \"loss_tokens\": n_loss,\n               \"seconds\": round(dt, 1), \"tasks\": [seqs[i][\"task\"] for i in grp]}\n        res[\"steps\"].append(rec)\n        res[\"tokens_per_second\"] = round(tok_done / (time.perf_counter() - t_train0), 1)\n        res[\"peak_gib_per_gpu\"] = gpu_peaks()\n        save()\n        log(f\"step {step + 1}/{total_steps} lr {lr:.2e} loss {rec['loss']} gnorm {rec['grad_norm']} tokens {n_tok} {dt:.0f}s | {res['tokens_per_second']} tok/s | peak {gpu_peaks()}\")\n        if (step + 1) % CKPT_EVERY == 0:\n            pm.save_pretrained(str(OUT / \"adapter_ckpt\"))\n            log(\"checkpoint saved\")\n    leak = max((torch.cuda.memory_allocated(i) - b) for i, b in enumerate(base_alloc)) / 2**30 if base_alloc else 0.0\n    res[\"leak_gib_at_end\"] = round(leak, 2)\n    res[\"stage\"] = \"saving\"\n    pm.save_pretrained(str(OUT / \"adapter_full\"))\n    res[\"stage\"] = \"done\"\n    res[\"train_seconds\"] = round(time.perf_counter() - t_train0, 1)\n    save()\n    log(\"training finished: steps\", len(res[\"steps\"]), \"of\", total_steps, \"| tok/s\", res.get(\"tokens_per_second\"), \"| adapter saved\")\n\n\nif __name__ == \"__main__\":\n    try:\n        main()\n    except BaseException as e:  # noqa: BLE001\n        res[\"stage\"] = \"FAILED\"\n        res[\"error\"] = f\"{type(e).__name__}: {str(e)[:500]}\"\n        res[\"traceback\"] = traceback.format_exc()[-3000:]\n        save()\n        traceback.print_exc()\n        sys.exit(1)\n", "g3_wiring.py": "\"\"\"Build the adapter evaluation arm and verify that a LoRA adapter really flows through the harness's compile path, BEFORE any GPU time is spent on training.\n\nmake_eval_arm: copy the B1 arm, add `adapter: <name>` right after the `model:` line of agent.yaml and sub_agents/code_analyzer.yaml, drop adapters/<name>/ in.\ncheck_wiring:  compile the arm exactly like swegemma 0.2.11 does (ModelRegistry with the base model and the adapter alias, discover_adapters manifest,\n               resolve_swegemma_adapter) and assert that the root agent's model resolves to the adapter alias (LiteLlm 'openai/<name>').\n\"\"\"\n\nimport re\nimport shutil\nfrom pathlib import Path\n\nADAPTER = \"g3_lora\"\nBASE = \"gemma-4-e4b-it-qat-w4a16-ct\"\n\n\ndef make_eval_arm(b1_dir: Path, adapter_src: Path, dst: Path, name: str = ADAPTER) -> Path:\n    shutil.rmtree(dst, ignore_errors=True)\n    shutil.copytree(b1_dir, dst)\n    for rel in (\"agent.yaml\", \"sub_agents/code_analyzer.yaml\"):\n        p = dst / rel\n        s = p.read_text(encoding=\"utf-8\")\n        pat = re.compile(rf\"^(model: {re.escape(BASE)}\\s*\\n)\", re.M)\n        assert len(pat.findall(s)) == 1, f\"{rel}: expected exactly one `model: {BASE}` line\"\n        assert \"adapter:\" not in s, f\"{rel} already declares an adapter\"\n        p.write_text(pat.sub(r\"\\1\" + f\"adapter: {name}\\n\", s, count=1), encoding=\"utf-8\", newline=\"\\n\")\n    (dst / \"adapters\" / name).mkdir(parents=True)\n    for f in (\"adapter_config.json\", \"adapter_model.safetensors\"):\n        shutil.copy(adapter_src / f, dst / \"adapters\" / name / f)\n    return dst\n\n\ndef check_wiring(arm: Path, name: str = ADAPTER) -> dict:\n    from adk_submission import ModelRegistry, compile_submission, discover_adapters, discover_declared_models\n    from google.adk.models.lite_llm import LiteLlm\n    from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS, build_submission_limits\n    from swegemma.context import SwegemmaContext\n    from swegemma.harness.agent_runner import resolve_swegemma_adapter\n    from swegemma.models import Task\n\n    task = Task(instance_id=\"d\", repo=\"d/d\", base_commit=\"0\" * 40, problem_statement=\"d\")\n    ctx = SwegemmaContext(docker_manager=None, container_id=\"\", problem_statement=\"d\", hints_text=\"\", task=task, repo=\"d/d\", graph_dir=\"/tmp\", embeddings_dir=\"/tmp\")\n    tools = ctx.create_tools()\n    declared = discover_declared_models(arm)\n    assert declared == {BASE}, declared\n    reg = ModelRegistry()\n    for m in declared:   # base model and adapter alias, as VllmServer.create_model_registry registers them when the server has an adapter manifest\n        reg.register(m, LiteLlm(model=\"openai/\" + m, api_base=\"http://127.0.0.1:8000/v1\", api_key=\"EMPTY\"))\n    manifest = discover_adapters(str(arm), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)\n    assert set(manifest.adapters) == {name}, manifest.adapters\n    reg.register(name, LiteLlm(model=\"openai/\" + name, api_base=\"http://127.0.0.1:8000/v1\", api_key=\"EMPTY\"))\n    limits, gen = build_submission_limits()\n    agent = compile_submission(submission_dir=arm, tool_registry=tools, model_registry=reg, limits=limits, generation_constraints=gen, adapter_manifest=manifest,\n                               adapter_resolver_fn=lambda b, i: resolve_swegemma_adapter(b, i, reg))\n    model = getattr(agent, \"model\", None)\n    model_id = getattr(model, \"model\", str(model))\n    assert model_id == f\"openai/{name}\", f\"root agent resolves to {model_id!r}, expected 'openai/{name}'\"\n    return {\"agent\": agent.name, \"root_model\": model_id, \"adapters\": sorted(manifest.adapters)}\n", "gemma4_memfix.py": "\"\"\"Memory fixes so a 31B Gemma 4 can be LoRA-trained on 4x L4 (23 GB each) at 16k+ tokens.\n\nRoot cause found in G1 run 4: full-attention layers have head_dim 512 (config `global_head_dim`), which PyTorch's fused SDPA kernels\ndo not support, so they fall back to the math kernel whose memory grows with T^2 ([32 heads, T, T] fp32). Even 5k tokens ran out of the\n~7 GiB per GPU left after loading 15.6 GiB of bf16 weights.\n\nFixes (all exact, no approximation of the model):\n  1. `chunked_sdpa` attention: queries processed in blocks, each block checkpointed, causal/sliding-window masks built on the fly from\n     positions (no T x T mask tensor). Memory ~ O(block x T) instead of O(T^2); sliding layers only look at their window.\n  2. Sequence-chunked MLP: the MLP is token-wise, so running it in chunks under checkpoint is exactly equivalent and removes the\n     [T, 21504] x several activations that dominate at 16k.\n\nAssumes batch size 1 and no padding (training on single unpadded sequences), which is how train_g1.py uses it.\n\"\"\"\n\nimport os\n\nimport torch\nfrom torch.utils.checkpoint import checkpoint\n\nATTN_NAME = \"chunked_sdpa\"\n_cfg = {\"chunk_global\": 256, \"chunk_sliding\": 1024, \"mlp_chunk\": 2048}\ncalls = {\"attn\": 0, \"mlp\": 0}  # incremented per call; lets a log prove which implementation actually ran\n\n\ndef _block(qc, kc, vc, q0: int, k0: int, scale: float, window):\n    \"\"\"qc [B,Hkv,G,c,D], kc/vc [B,Hkv,L,D] -> [B,Hkv,G,c,D]; exact causal (+ sliding-window) softmax attention in fp32 scores.\"\"\"\n    s = torch.einsum(\"bhgcd,bhld->bhgcl\", qc, kc).float() * scale\n    qi = torch.arange(q0, q0 + qc.shape[3], device=qc.device)[:, None]\n    kj = torch.arange(k0, k0 + kc.shape[2], device=qc.device)[None, :]\n    allowed = kj <= qi\n    if window:\n        allowed = allowed & (kj > qi - window)\n    s = s.masked_fill(~allowed, float(\"-inf\"))\n    p = torch.softmax(s, dim=-1).to(vc.dtype)\n    return torch.einsum(\"bhgcl,bhld->bhgcd\", p, vc)\n\n\ndef chunked_attention(module, query, key, value, attention_mask, dropout=0.0, scaling=None, sliding_window=None, **kwargs):\n    \"\"\"Drop-in for transformers attention functions: query [B,H,T,D], key/value [B,Hkv,T,D] -> ([B,T,H,D], None).\"\"\"\n    calls[\"attn\"] += 1\n    B, H, T, D = query.shape\n    Hkv = key.shape[1]\n    G = H // Hkv\n    scale = 1.0 if scaling is None else scaling\n    ch = _cfg[\"chunk_sliding\"] if sliding_window else _cfg[\"chunk_global\"]\n    q = query.reshape(B, Hkv, G, T, D)\n    outs = []\n    for a in range(0, T, ch):\n        b = min(a + ch, T)\n        k0 = max(0, a - sliding_window + 1) if sliding_window else 0\n        args = (q[:, :, :, a:b], key[:, :, k0:b], value[:, :, k0:b], a, k0, scale, sliding_window)\n        outs.append(checkpoint(_block, *args, use_reentrant=False) if torch.is_grad_enabled() else _block(*args))\n    out = torch.cat(outs, dim=3).reshape(B, H, T, D)\n    return out.transpose(1, 2).contiguous(), None\n\n\ndef _no_mask(*args, **kwargs):\n    return None  # chunked_attention builds its own causal / sliding-window masks\n\n\ndef install(chunk_global: int = 256, chunk_sliding: int = 1024, mlp_chunk: int = 2048) -> None:\n    \"\"\"Register the attention + mask functions and patch the Gemma 4 MLP. Call before building/loading the model.\"\"\"\n    from transformers import AttentionInterface\n    from transformers.masking_utils import AttentionMaskInterface\n    from transformers.models.gemma4 import modeling_gemma4 as mg\n\n    _cfg.update(chunk_global=chunk_global, chunk_sliding=chunk_sliding, mlp_chunk=mlp_chunk)\n    AttentionInterface.register(ATTN_NAME, chunked_attention)\n    AttentionMaskInterface.register(ATTN_NAME, _no_mask)\n\n    if not getattr(mg.Gemma4TextMLP, \"_g1_chunked\", False):\n        orig = mg.Gemma4TextMLP.forward\n\n        def forward(self, x):\n            n = x.shape[-2]\n            mc = _cfg[\"mlp_chunk\"]\n            if n <= mc:\n                return orig(self, x)\n            calls[\"mlp\"] += 1\n            parts = [checkpoint(orig, self, x[..., a:a + mc, :], use_reentrant=False) if torch.is_grad_enabled() else orig(self, x[..., a:a + mc, :])\n                     for a in range(0, n, mc)]\n            return torch.cat(parts, dim=-2)\n\n        mg.Gemma4TextMLP.forward = forward\n        mg.Gemma4TextMLP._g1_chunked = True\n        mg.Gemma4TextMLP._g1_orig_forward = orig\n\n\ndef uninstall_mlp_patch() -> None:\n    \"\"\"For tests: restore the stock MLP forward.\"\"\"\n    from transformers.models.gemma4 import modeling_gemma4 as mg\n    if getattr(mg.Gemma4TextMLP, \"_g1_chunked\", False):\n        mg.Gemma4TextMLP.forward = mg.Gemma4TextMLP._g1_orig_forward\n        mg.Gemma4TextMLP._g1_chunked = False\n\n\n# =====================================================================================================================\n# Sequence-chunked decoder layer (G1d). Everything in a Gemma 4 layer is token-wise EXCEPT attention itself, so a layer can be\n# processed in chunks of query positions with each chunk checkpointed. The big fp32 temporaries of RMSNorm / q-norm / RoPE / MLP\n# then scale with the chunk instead of the whole sequence. Sliding layers recompute the K/V of their window per chunk (cheap);\n# global layers compute K/V once (checkpointed in token chunks; only ~[T, 2048] per tensor to keep).\n# Valid for the 31B config: no per-layer inputs, no MoE, no KV sharing, batch 1, no cache. Anything else falls back to stock.\n# =====================================================================================================================\n_lcfg = {\"chunk_sliding\": 1024, \"chunk_global\": 256, \"kv_chunk\": 2048, \"min_len\": 2048}\n\n\ndef _kv(layer, h, cos, sin):\n    from transformers.models.gemma4.modeling_gemma4 import apply_rotary_pos_emb\n    a = layer.self_attn\n    x = layer.input_layernorm(h)\n    shape = (*x.shape[:-1], -1, a.head_dim)\n    k_raw = a.k_proj(x).view(shape)\n    v_raw = a.v_proj(x).view(shape) if a.v_proj is not None else k_raw   # k_eq_v layers: value = raw (pre-norm) key projection\n    k = apply_rotary_pos_emb(a.k_norm(k_raw), cos, sin, unsqueeze_dim=2).transpose(1, 2)\n    v = a.v_norm(v_raw).transpose(1, 2)\n    return k, v\n\n\ndef _tail(layer, h_c, cos_c, sin_c, K, V, q0, k0):\n    \"\"\"The whole layer for the query tokens h_c (positions q0..q0+c): attention against K/V (positions k0..) + MLP, exactly as stock.\"\"\"\n    from transformers.models.gemma4.modeling_gemma4 import apply_rotary_pos_emb\n    a = layer.self_attn\n    c = h_c.shape[1]\n    x = layer.input_layernorm(h_c)\n    q = apply_rotary_pos_emb(a.q_norm(a.q_proj(x).view(1, c, -1, a.head_dim)), cos_c, sin_c, unsqueeze_dim=2).transpose(1, 2)  # [1,H,c,hd]\n    H, Hkv = q.shape[1], K.shape[1]\n    o = _block(q.reshape(1, Hkv, H // Hkv, c, q.shape[-1]), K, V, q0, k0, a.scaling, a.sliding_window)\n    o = a.o_proj(o.reshape(1, H, c, -1).transpose(1, 2).reshape(1, c, -1))\n    h = h_c + layer.post_attention_layernorm(o)\n    h = h + layer.post_feedforward_layernorm(layer.mlp(layer.pre_feedforward_layernorm(h)))\n    return h * layer.layer_scalar\n\n\ndef _chunk_sliding(layer, h_kv, cos_kv, sin_kv, q0, k0):\n    off = q0 - k0\n    K, V = _kv(layer, h_kv, cos_kv, sin_kv)\n    return _tail(layer, h_kv[:, off:], cos_kv[:, off:], sin_kv[:, off:], K, V, q0, k0)\n\n\ndef _chunk_global(layer, h_c, cos_c, sin_c, K, V, q0):\n    b = q0 + h_c.shape[1]\n    return _tail(layer, h_c, cos_c, sin_c, K[:, :, :b], V[:, :, :b], q0, 0)\n\n\ndef _layer_forward(self, hidden_states, per_layer_input=None, shared_kv_states=None, position_embeddings=None, attention_mask=None,\n                   position_ids=None, past_key_values=None, **kwargs):\n    B, T, _ = hidden_states.shape\n    a = self.self_attn\n    if (B != 1 or T < _lcfg[\"min_len\"] or past_key_values is not None or not torch.is_grad_enabled() or self.enable_moe_block\n            or self.hidden_size_per_layer_input or a.is_kv_shared_layer):\n        return type(self)._g1_orig_forward(self, hidden_states, per_layer_input, shared_kv_states, position_embeddings, attention_mask,\n                                           position_ids, past_key_values, **kwargs)\n    calls[\"layer\"] += 1\n    cos, sin = position_embeddings\n    h = hidden_states\n    outs = []\n    if a.sliding_window:\n        ch, w = _lcfg[\"chunk_sliding\"], a.sliding_window\n        for q0 in range(0, T, ch):\n            b, k0 = min(q0 + ch, T), max(0, q0 - w + 1)\n            outs.append(checkpoint(_chunk_sliding, self, h[:, k0:b], cos[:, k0:b], sin[:, k0:b], q0, k0, use_reentrant=False))\n    else:\n        ks, vs = [], []\n        for t0 in range(0, T, _lcfg[\"kv_chunk\"]):\n            t1 = min(t0 + _lcfg[\"kv_chunk\"], T)\n            k, v = checkpoint(_kv, self, h[:, t0:t1], cos[:, t0:t1], sin[:, t0:t1], use_reentrant=False)\n            ks.append(k)\n            vs.append(v)\n        K, V = torch.cat(ks, dim=2), torch.cat(vs, dim=2)\n        ch = _lcfg[\"chunk_global\"]\n        for q0 in range(0, T, ch):\n            b = min(q0 + ch, T)\n            outs.append(checkpoint(_chunk_global, self, h[:, q0:b], cos[:, q0:b], sin[:, q0:b], K, V, q0, use_reentrant=False))\n    return torch.cat(outs, dim=1)\n\n\ncalls[\"layer\"] = 0\n\n\ndef install_layers(chunk_sliding: int = 1024, chunk_global: int = 256, kv_chunk: int = 2048, min_len: int = 2048) -> None:\n    \"\"\"Replace Gemma4TextDecoderLayer.forward by the sequence-chunked version (use with HF gradient checkpointing DISABLED:\n    the chunks are the checkpoints).\"\"\"\n    from transformers.models.gemma4 import modeling_gemma4 as mg\n    _lcfg.update(chunk_sliding=chunk_sliding, chunk_global=chunk_global, kv_chunk=kv_chunk, min_len=min_len)\n    L = mg.Gemma4TextDecoderLayer\n    if not getattr(L, \"_g1_layer_chunked\", False):\n        L._g1_orig_forward = L.forward\n        L.forward = _layer_forward\n        L._g1_layer_chunked = True\n\n\ndef uninstall_layers() -> None:\n    from transformers.models.gemma4 import modeling_gemma4 as mg\n    L = mg.Gemma4TextDecoderLayer\n    if getattr(L, \"_g1_layer_chunked\", False):\n        L.forward = L._g1_orig_forward\n        L._g1_layer_chunked = False\n", "g1_util.py": "\"\"\"Utilities for the G1 notebook, tested locally (test_g1_util.py) before use on paid GPU time.\n\nstage_checkpoint: copy a checkpoint dir from the slow NFS mount (/kaggle/input: ~90 MB/s per stream, ~320 MB/s with 4 parallel streams)\n                  to local disk with many parallel large-block reads; aborts early on poor throughput.\nrun_supervised:   run a subprocess and kill it if it misses progress milestones (instead of a long fixed timeout that burns GPU quota).\n\"\"\"\n\nimport concurrent.futures as cf\nimport os\nimport shutil\nimport subprocess\nimport threading\nimport time\nfrom pathlib import Path\n\n\nclass StageAbort(RuntimeError):\n    pass\n\n\ndef _copy_block(src: Path, dst: Path, off: int, n: int) -> int:\n    with open(src, \"rb\", buffering=0) as fi, open(dst, \"r+b\", buffering=0) as fo:\n        fi.seek(off)\n        fo.seek(off)\n        left = n\n        while left:\n            buf = fi.read(left)\n            if not buf:\n                raise IOError(f\"short read from {src} at {off + n - left}\")\n            fo.write(buf)\n            left -= len(buf)\n    return n\n\n\ndef stage_checkpoint(src: Path, dst: Path, workers: int = 12, block: int = 64 << 20, min_mbps_after_s: float = 180, min_mbps: float = 30,\n                     max_seconds: float = 1500, log=print) -> dict:\n    dst.mkdir(parents=True, exist_ok=True)\n    jobs, total = [], 0\n    for f in sorted(src.iterdir()):\n        if not f.is_file():\n            continue\n        size = f.stat().st_size\n        total += size\n        if size < block:\n            shutil.copy(f, dst / f.name)\n            continue\n        with open(dst / f.name, \"wb\") as o:\n            o.truncate(size)\n        jobs += [(f, dst / f.name, off, min(block, size - off)) for off in range(0, size, block)]\n    t0, done = time.time(), 0\n    with cf.ThreadPoolExecutor(workers) as ex:\n        futs = [ex.submit(_copy_block, *j) for j in jobs]\n        pending = set(futs)\n        last_log = 0.0\n        while pending:\n            fin, pending = cf.wait(pending, timeout=5, return_when=cf.FIRST_COMPLETED)\n            for fu in fin:\n                done += fu.result()  # re-raises read errors\n            el = time.time() - t0\n            if el - last_log >= 20:\n                last_log = el\n                log(f\"[stage] {done / 1e9:.1f}/{sum(j[3] for j in jobs) / 1e9:.1f} GB, {done / 1e6 / max(el, 1e-9):.0f} MB/s, {el:.0f}s\")\n            if el > min_mbps_after_s and done / 1e6 / el < min_mbps:\n                ex.shutdown(wait=False, cancel_futures=True)\n                raise StageAbort(f\"throughput {done / 1e6 / el:.0f} MB/s < {min_mbps} after {el:.0f}s\")\n            if el > max_seconds:\n                ex.shutdown(wait=False, cancel_futures=True)\n                raise StageAbort(f\"staging exceeded {max_seconds}s\")\n    el = time.time() - t0\n    return {\"bytes\": total, \"seconds\": round(el, 1), \"MBps\": round(sum(j[3] for j in jobs) / 1e6 / max(el, 1e-9), 1), \"workers\": workers}\n\n\ndef run_supervised(cmd: list[str], env: dict, logfile: Path, milestones: list[tuple[str, float]], max_seconds: float, poll: float = 10) -> dict:\n    \"\"\"milestones: [(substring that must appear in the log, deadline in seconds since start)]. Missing a deadline kills the process.\"\"\"\n    t0 = time.time()\n    with open(logfile, \"w\") as lf:\n        p = subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT)\n        reached: dict[str, float] = {}\n        verdict = \"finished\"\n        while p.poll() is None:\n            time.sleep(poll)\n            el = time.time() - t0\n            txt = Path(logfile).read_text(errors=\"ignore\")\n            for pat, deadline in milestones:\n                if pat not in reached and pat in txt:\n                    reached[pat] = round(el, 1)\n            missed = [pat for pat, deadline in milestones if pat not in reached and el > deadline]\n            if missed:\n                verdict = f\"killed: milestone '{missed[0]}' not reached by {dict(milestones)[missed[0]]:.0f}s\"\n                break\n            if el > max_seconds:\n                verdict = f\"killed: exceeded {max_seconds:.0f}s\"\n                break\n        if p.poll() is None:\n            p.kill()\n            p.wait()\n    return {\"returncode\": p.returncode, \"verdict\": verdict, \"seconds\": round(time.time() - t0, 1), \"milestones_reached\": reached}\n", "make_adapters.py": "\"\"\"Turn the raw PEFT output of train_g1.py into scorer-format adapter directories and verify them against the sample adapter format.\n\nVariants written under <out>/:\n  g1_full/    all trained LoRA matrices (attention + MLP)\n  g1_attn/    attention-only subset (q,k,v,o): the safest set for vLLM LoRA support\n  g1_canary/  attention-only with RANDOM lora_B (so the adapter visibly changes outputs even after only a few training steps):\n              used only to prove the serving path really applies adapters (\"silently zero\" LoRA was reported by other teams)\n\nEvery variant: bf16 tensors, keys `base_model.model.model.language_model.layers.N.<module>.lora_{A,B}.weight`, adapter_config.json with\nlist-valued target_modules and base_model_name_or_path = the scorer's model id.\n\nUsage: python make_adapters.py <adapter_full_dir> <out_dir> [sample_adapter_dir]\n\"\"\"\n\nimport json\nimport re\nimport sys\nfrom pathlib import Path\n\nimport torch\nfrom safetensors.torch import load_file, save_file\n\nimport os\nSCORER_MODEL = os.environ.get(\"SCORER_MODEL\", \"google/gemma-4-31b-it-qat-w4a16-ct\")   # e.g. google/gemma-4-e4b-it-qat-w4a16-ct\nKEY_RE = re.compile(r\"^base_model\\.model\\.model\\.language_model\\.layers\\.(\\d+)\\.(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|mlp\\.(gate_proj|up_proj|down_proj))\\.lora_(A|B)\\.weight$\")\nATTN = (\"q_proj\", \"k_proj\", \"v_proj\", \"o_proj\")\n\n\ndef write_variant(out: Path, tensors: dict[str, torch.Tensor], cfg: dict) -> dict:\n    out.mkdir(parents=True, exist_ok=True)\n    mods = sorted({k.split(\".\")[-3] for k in tensors})\n    cfg = dict(cfg, base_model_name_or_path=SCORER_MODEL, inference_mode=True, target_modules=mods)\n    cfg.pop(\"layers_to_transform\", None) if cfg.get(\"layers_to_transform\") is None else None\n    (out / \"adapter_config.json\").write_text(json.dumps(cfg, indent=2), encoding=\"utf-8\")\n    save_file({k: v.to(torch.bfloat16).contiguous() for k, v in tensors.items()}, str(out / \"adapter_model.safetensors\"), metadata={\"format\": \"pt\"})\n    size = (out / \"adapter_model.safetensors\").stat().st_size\n    return {\"dir\": str(out), \"tensors\": len(tensors), \"target_modules\": mods, \"bytes\": size}\n\n\ndef main(src: Path, out: Path, sample: Path | None) -> dict:\n    t = load_file(str(src / \"adapter_model.safetensors\"))\n    cfg = json.loads((src / \"adapter_config.json\").read_text(encoding=\"utf-8\"))\n    bad = [k for k in t if not KEY_RE.match(k)]\n    assert not bad, f\"unexpected tensor names, e.g. {bad[:3]}\"\n    report: dict = {\"n_tensors\": len(t), \"rank\": cfg.get(\"r\"), \"alpha\": cfg.get(\"lora_alpha\"), \"dtype_in\": str(next(iter(t.values())).dtype)}\n    if sample is not None and (sample / \"adapter_model.safetensors\").exists():\n        sk = list(load_file(str(sample / \"adapter_model.safetensors\")))\n        report[\"sample_key_example\"] = sk[0]\n        report[\"our_key_example\"] = sorted(t)[0]\n        # same naming scheme (prefix up to the layer index) as the scorer's sample adapter\n        pre = lambda k: k.split(\".layers.\")[0]  # noqa: E731\n        assert pre(sk[0]) == pre(sorted(t)[0]), (pre(sk[0]), pre(sorted(t)[0]))\n        sample_cfg = json.loads((sample / \"adapter_config.json\").read_text(encoding=\"utf-8\"))\n        report[\"sample_cfg_keys_missing_in_ours\"] = sorted(set(sample_cfg) - set(cfg))\n    report[\"g1_full\"] = write_variant(out / \"g1_full\", t, cfg)\n    attn = {k: v for k, v in t.items() if k.split(\".\")[-3] in ATTN}\n    report[\"g1_attn\"] = write_variant(out / \"g1_attn\", attn, cfg)\n    g = torch.Generator().manual_seed(0)\n    canary = {k: (torch.randn(v.shape, generator=g) * 0.02 if k.endswith(\"lora_B.weight\") else v.clone()) for k, v in attn.items()}\n    report[\"g1_canary\"] = write_variant(out / \"g1_canary\", canary, cfg)\n    (out / \"adapters_report.json\").write_text(json.dumps(report, indent=1), encoding=\"utf-8\")\n    return report\n\n\nif __name__ == \"__main__\":\n    r = main(Path(sys.argv[1]), Path(sys.argv[2]), Path(sys.argv[3]) if len(sys.argv) > 3 else None)\n    print(json.dumps(r, indent=1))\n", "capture_proxy.py": "\"\"\"Transparent logging reverse proxy between the harness (LiteLLM -> ADK) and vLLM, for G2 data collection.\n\nFor every non-streaming POST .../chat/completions it records, gzip-compressed, one JSON line with the FULL request body (messages incl. reasoning,\ntools, chat_template_kwargs, sampling params) and the FULL response body. It asks vLLM for `return_token_ids` so the record also carries the exact\nprompt token ids and completion token ids (no re-rendering guesses); if the server rejects that parameter it retries without it and keeps going.\nThe harness receives the response with the token-id fields stripped, i.e. exactly what it would get without the proxy.\nEverything else (GET /v1/models, /health, streaming requests) is passed through untouched.\n\"\"\"\n\nimport gzip\nimport hashlib\nimport json\nimport threading\nimport time\nfrom http.server import BaseHTTPRequestHandler, ThreadingHTTPServer\n\nimport requests\n\n\ndef task_key(messages: list[dict]) -> str:\n    \"\"\"Stable id of the agent session a request belongs to: hash of the system prompt + the first user message (the task statement).\"\"\"\n    sys_txt = next((m.get(\"content\") for m in messages if m.get(\"role\") == \"system\"), \"\") or \"\"\n    usr_txt = next((m.get(\"content\") for m in messages if m.get(\"role\") == \"user\"), \"\") or \"\"\n    h = hashlib.sha1()\n    for part in (sys_txt, usr_txt):\n        h.update((part if isinstance(part, str) else json.dumps(part, sort_keys=True)).encode(\"utf-8\", \"ignore\"))\n        h.update(b\"\\x00\")\n    return h.hexdigest()[:16]\n\n\nclass CaptureProxy:\n    def __init__(self, upstream: str, out_path: str, port: int = 8001, host: str = \"127.0.0.1\", ask_token_ids: bool = True):\n        self.upstream = upstream.rstrip(\"/\")          # e.g. http://127.0.0.1:8000\n        self.out_path = out_path\n        self.port, self.host = port, host\n        self.ask_token_ids = ask_token_ids\n        self.round_tag = \"r0\"\n        self.tokenize_fallback = False        # if True and vLLM returns no token ids, ask /tokenize for the exact prompt ids instead\n        self.n_records = 0\n        self.n_with_ids = 0\n        self.n_errors = 0\n        self._lock = threading.Lock()\n        self._httpd: ThreadingHTTPServer | None = None\n\n    # ---------------------------------------------------------------- lifecycle\n    def start(self) -> \"CaptureProxy\":\n        proxy = self\n\n        class H(BaseHTTPRequestHandler):\n            protocol_version = \"HTTP/1.1\"\n\n            def log_message(self, *a):  # silence\n                pass\n\n            def _relay(self, status: int, body: bytes, ctype: str = \"application/json\"):\n                self.send_response(status)\n                self.send_header(\"Content-Type\", ctype)\n                self.send_header(\"Content-Length\", str(len(body)))\n                self.end_headers()\n                self.wfile.write(body)\n\n            def do_GET(self):\n                try:\n                    r = requests.get(proxy.upstream + self.path, timeout=60)\n                    self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n                except Exception as e:  # noqa: BLE001\n                    self._relay(502, json.dumps({\"error\": str(e)}).encode())\n\n            def do_POST(self):\n                raw = self.rfile.read(int(self.headers.get(\"Content-Length\", \"0\")))\n                try:\n                    req = json.loads(raw) if raw else {}\n                except Exception:  # noqa: BLE001\n                    req = None\n                is_chat = self.path.rstrip(\"/\").endswith(\"/chat/completions\") and isinstance(req, dict) and not req.get(\"stream\")\n                if not is_chat:                      # pass-through (streaming, other endpoints)\n                    try:\n                        r = requests.post(proxy.upstream + self.path, data=raw, headers={\"Content-Type\": \"application/json\"}, timeout=1800)\n                        self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n                    except Exception as e:  # noqa: BLE001\n                        self._relay(502, json.dumps({\"error\": str(e)}).encode())\n                    return\n                t0 = time.time()\n                asked = proxy.ask_token_ids           # decided ONCE per request (another thread may flip the flag meanwhile)\n                body = dict(req, return_token_ids=True) if asked else req\n                try:\n                    r = requests.post(proxy.upstream + self.path, json=body, timeout=1800)\n                    if asked and r.status_code == 400 and \"return_token_ids\" in r.text:\n                        proxy.ask_token_ids = False   # this vLLM rejects the parameter: stop asking, redo this call plainly\n                        r = requests.post(proxy.upstream + self.path, json=req, timeout=1800)\n                except Exception as e:  # noqa: BLE001\n                    proxy.n_errors += 1\n                    self._relay(502, json.dumps({\"error\": f\"proxy upstream failure: {e}\"}).encode())\n                    return\n                try:\n                    resp = r.json()\n                except Exception:  # noqa: BLE001\n                    resp = None\n                if r.status_code == 200 and isinstance(resp, dict):\n                    proxy._record(req, resp, t0, time.time())\n                    clean = {k: v for k, v in resp.items() if k != \"prompt_token_ids\"}\n                    clean[\"choices\"] = [{k: v for k, v in ch.items() if k != \"token_ids\"} for ch in resp.get(\"choices\", [])]\n                    self._relay(200, json.dumps(clean).encode())\n                else:\n                    proxy.n_errors += 1\n                    self._relay(r.status_code, r.content, r.headers.get(\"Content-Type\", \"application/json\"))\n\n        self._httpd = ThreadingHTTPServer((self.host, self.port), H)\n        self._httpd.daemon_threads = True\n        threading.Thread(target=self._httpd.serve_forever, daemon=True).start()\n        return self\n\n    def stop(self) -> None:\n        if self._httpd:\n            self._httpd.shutdown()\n            self._httpd.server_close()\n\n    # ---------------------------------------------------------------- recording\n    def _record(self, req: dict, resp: dict, t0: float, t1: float) -> None:\n        has_ids = bool(resp.get(\"prompt_token_ids\")) and bool((resp.get(\"choices\") or [{}])[0].get(\"token_ids\"))\n        rec = {\"round\": self.round_tag, \"task_key\": task_key(req.get(\"messages\", [])), \"t0\": round(t0, 3), \"t1\": round(t1, 3), \"request\": req, \"response\": resp}\n        if not has_ids and self.tokenize_fallback:\n            try:\n                tk = {k: req[k] for k in (\"model\", \"messages\", \"tools\", \"chat_template_kwargs\") if k in req}\n                tk[\"add_generation_prompt\"] = True\n                r = requests.post(self.upstream + \"/tokenize\", json=tk, timeout=300)\n                if r.status_code == 200:\n                    rec[\"prompt_token_ids_tokenize\"] = r.json().get(\"tokens\")\n            except Exception:  # noqa: BLE001\n                pass\n        line = (json.dumps(rec, ensure_ascii=False) + \"\\n\").encode(\"utf-8\")\n        with self._lock:                             # one gzip member per record; the concatenation is a valid gzip stream\n            with open(self.out_path, \"ab\") as f:\n                f.write(gzip.compress(line, compresslevel=3))\n            self.n_records += 1\n            self.n_with_ids += has_ids\n\n    def stats(self) -> dict:\n        return {\"records\": self.n_records, \"with_token_ids\": self.n_with_ids, \"errors\": self.n_errors, \"asking_token_ids\": self.ask_token_ids, \"round\": self.round_tag}\n\n\ndef read_records(path: str):\n    \"\"\"Iterate the records of a capture file.\"\"\"\n    with gzip.open(path, \"rt\", encoding=\"utf-8\") as f:\n        for line in f:\n            if line.strip():\n                yield json.loads(line)\n"}.items(): (CODE / n).write_text(txt, encoding='utf-8')
for n, txt in {"train_opd.py": "\"\"\"G21: LoRA on-policy distillation (OPD) of a Gemma 4 student from teacher top-k log-probs. Derived from scripts/g3/train_g3.py (same memory layout,\noptimiser, schedule, checkpointing and G3_* environment variables); ONLY the per-token loss and the input fields differ.\n\nInput jsonl rows (scripts/g20/build_opd_dataset.py): {\"input_ids\", \"loss_spans\", \"n_loss_tokens\", \"task\", \"t_ids\": [[k ids], ...], \"t_lp\": [[k log-probs], ...]}\nwith one teacher row per target position j inside the spans, in increasing j (the student predicts token j from position j-1, as in train_g3).\nThe loss per position is the exact KL between student and teacher coarse-grained to the teacher's top-k tokens + one bucket for the rest of the vocabulary\n(scripts/g20/opd_loss.py; reverse KL by default, OPD_DIV=fkl for forward KL); groups are normalised by their total number of loss positions.\nThe sequences are the student's OWN samples (captured on-policy), so this is the first iteration of on-policy distillation.\nMemory: sequence-chunked decoder layers (scripts/g1/gemma4_memfix.py, exact, verified at the real 31B dimensions) + chunked lm_head loss: trained and probed\non the L4x4 up to 18,432 tokens in G1d. The environment fixes found in G1 are kept: torchao hidden from peft, checkpoint staged to local disk by the notebook.\n\nEnvironment variables: G3_MODEL, G3_DATA, G3_OUT (required); G3_TINY=1 (tiny random model for local tests); G3_ACC (default 2); G3_LR (2e-4);\nG3_EPOCHS (1); G3_HF_GC (1 = HF gradient checkpointing, for E4B/E2B); G3_TINY_PLE (1 = tiny test model with per-layer inputs); G3_WARMUP (3 optimizer steps); G3_SEED (0); G3_MAXMEM (JSON per-GPU limits); G3_MAX_SEQS (use only the first N sequences, tests); G3_CKPT_EVERY (10).\nOPD_DIV (rkl | fkl, default rkl).\n\"\"\"\n\nimport contextlib\nimport gc\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\nimport traceback\nfrom pathlib import Path\n\nos.environ.setdefault(\"PYTORCH_CUDA_ALLOC_CONF\", \"expandable_segments:True\")\nsys.modules[\"torchao\"] = None   # Kaggle's torchao 0.10.0 is rejected by peft 0.19.1 and we do not use it\nimport torch\nimport torch.nn.functional as F\nimport transformers\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nsys.path.insert(0, str(Path(__file__).resolve().parent.parent / \"g1\"))\nimport gemma4_memfix as MF  # noqa: E402\nimport opd_loss as OL  # noqa: E402\n\nMODEL, DATA, OUT = os.environ[\"G3_MODEL\"], os.environ[\"G3_DATA\"], Path(os.environ[\"G3_OUT\"])\nTINY = os.environ.get(\"G3_TINY\") == \"1\"\nACC = int(os.environ.get(\"G3_ACC\", \"2\"))\nLR = float(os.environ.get(\"G3_LR\", \"2e-4\"))\nEPOCHS = int(os.environ.get(\"G3_EPOCHS\", \"1\"))\nWARMUP = int(os.environ.get(\"G3_WARMUP\", \"3\"))\nSEED = int(os.environ.get(\"G3_SEED\", \"0\"))\nMAXMEM = json.loads(os.environ.get(\"G3_MAXMEM\", \"null\"))\nMAX_SEQS = int(os.environ.get(\"G3_MAX_SEQS\", \"0\"))\nCKPT_EVERY = int(os.environ.get(\"G3_CKPT_EVERY\", \"10\"))\nTEST_MEMFRAC = os.environ.get(\"G3_TEST_MEMFRAC\")\nHF_GC = os.environ.get(\"G3_HF_GC\") == \"1\"       # HF gradient checkpointing: needed for models whose layers fall back to the stock forward (E4B/E2B: per-layer inputs, shared KV)\nTINY_PLE = os.environ.get(\"G3_TINY_PLE\") == \"1\"  # tiny test model WITH per-layer inputs and shared-KV layers (E4B-like)\nDIV = os.environ.get(\"OPD_DIV\", \"rkl\")\nassert DIV in (\"rkl\", \"fkl\"), DIV\nTARGET_RE = r\".*language_model\\.layers\\.\\d+\\.(self_attn\\.(q_proj|k_proj|v_proj|o_proj)|mlp\\.(gate_proj|up_proj|down_proj))\"\nOUT.mkdir(parents=True, exist_ok=True)\nres: dict = {\"stage\": \"start\", \"objective\": f\"opd_{DIV}\", \"tiny\": TINY, \"acc\": ACC, \"lr\": LR, \"epochs\": EPOCHS, \"warmup\": WARMUP, \"seed\": SEED,\n             \"torch\": torch.__version__, \"transformers\": transformers.__version__, \"n_gpus\": torch.cuda.device_count()}\n\n\ndef save() -> None:\n    (OUT / \"train_metrics.json\").write_text(json.dumps(res, indent=1), encoding=\"utf-8\")\n\n\ndef log(*a) -> None:\n    print(\"[opd]\", *a, flush=True)\n\n\ndef gpu_peaks() -> list[float]:\n    return [round(torch.cuda.max_memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())]\n\n\ndef load_model():\n    cls = transformers.Gemma4ForConditionalGeneration\n    MF.install()\n    MF.install_layers(chunk_sliding=1024, chunk_global=128)\n    if TINY:\n        torch.manual_seed(SEED)   # deterministic tiny model: the test rebuilds it in-process to make a teacher equal to the student\n        from transformers import Gemma4Config, Gemma4TextConfig\n        t = Gemma4TextConfig(vocab_size=1000, hidden_size=64, intermediate_size=128, num_hidden_layers=6, num_attention_heads=4, num_key_value_heads=2,\n                             head_dim=16, sliding_window=32, layer_types=([\"sliding_attention\", \"sliding_attention\", \"full_attention\", \"sliding_attention\", \"sliding_attention\", \"full_attention\"] if TINY_PLE\n                                     else [\"sliding_attention\"] * 5 + [\"full_attention\"]), final_logit_softcapping=30.0,\n                             hidden_size_per_layer_input=(16 if TINY_PLE else 0), vocab_size_per_layer_input=1000, num_kv_shared_layers=(2 if TINY_PLE else 0),\n                             use_double_wide_mlp=False, max_position_embeddings=65536)\n        cfg = Gemma4Config(text_config=t.to_dict(), vision_config=dict(hidden_size=32, intermediate_size=64, num_hidden_layers=1, num_attention_heads=2,\n                           num_key_value_heads=2, head_dim=16, global_head_dim=16), audio_config=None)\n        return cls._from_config(cfg, attn_implementation=\"sdpa\").to(torch.bfloat16).to(\"cuda:0\" if torch.cuda.is_available() else \"cpu\")\n    kw = dict(dtype=torch.bfloat16, device_map=\"auto\", attn_implementation=\"sdpa\")\n    if MAXMEM:\n        kw[\"max_memory\"] = {int(k): v for k, v in MAXMEM.items()}\n    return cls.from_pretrained(MODEL, **kw)\n\n\ndef main() -> None:\n    seqs = [json.loads(line) for line in open(DATA, encoding=\"utf-8\")]\n    if MAX_SEQS:\n        seqs = seqs[:MAX_SEQS]\n    for s in seqs:   # the teacher rows must cover exactly the target positions of the spans (the same set fwd_bwd builds)\n        n = len({j for a, b in s[\"loss_spans\"] for j in range(max(a, 1), min(b, len(s[\"input_ids\"])))})\n        assert n == s[\"n_loss_tokens\"] == len(s[\"t_ids\"]) == len(s[\"t_lp\"]), (s[\"task\"], n, s[\"n_loss_tokens\"], len(s[\"t_ids\"]))\n    res[\"data\"] = {\"sequences\": len(seqs), \"tokens\": sum(len(s[\"input_ids\"]) for s in seqs), \"loss_tokens\": sum(s[\"n_loss_tokens\"] for s in seqs),\n                   \"max_len\": max(len(s[\"input_ids\"]) for s in seqs), \"tasks\": len({s[\"task\"] for s in seqs})}\n    log(\"data\", res[\"data\"])\n    save()\n\n    t0 = time.perf_counter()\n    model = load_model()\n    res[\"load_seconds\"] = round(time.perf_counter() - t0, 1)\n    res[\"mem_after_load_gib\"] = [round(torch.cuda.memory_allocated(i) / 2**30, 2) for i in range(torch.cuda.device_count())]\n    log(\"loaded in\", res[\"load_seconds\"], \"s; mem after load (GiB):\", res[\"mem_after_load_gib\"])\n    save()\n\n    if TEST_MEMFRAC and torch.cuda.is_available():\n        torch.cuda.set_per_process_memory_fraction(float(TEST_MEMFRAC))\n    from peft import LoraConfig, get_peft_model\n    model.config.use_cache = False\n    if HF_GC:\n        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\"use_reentrant\": False})\n        log(\"HF gradient checkpointing enabled\")\n    pm = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, bias=\"none\", target_modules=TARGET_RE, task_type=\"CAUSAL_LM\"))\n    params = [p for p in pm.parameters() if p.requires_grad]\n    res[\"trainable_params_m\"] = round(sum(p.numel() for p in params) / 1e6, 1)\n    assert any(n.endswith(\".lora_A.default\") for n, _ in pm.named_modules()), \"TARGET_RE matched no module\"\n    pm.train()\n    dev = model.get_input_embeddings().weight.device\n    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)\n    inner, lm_head = model.model, model.lm_head\n    softcap = getattr(model.config.get_text_config(), \"final_logit_softcapping\", None)\n    from torch.utils.checkpoint import checkpoint\n    CH = 512\n    base_alloc = [torch.cuda.memory_allocated(i) for i in range(torch.cuda.device_count())]\n\n    def chunk_loss(h, t_ids, t_lp):\n        logits = lm_head(h).float()\n        if softcap:\n            logits = torch.tanh(logits / softcap) * softcap\n        return OL.coarse_kl(logits, t_ids, t_lp, DIV).sum()\n\n    def fwd_bwd(seq: dict, denom: float) -> tuple[float, int]:\n        ids = seq[\"input_ids\"]\n        pos = sorted({j - 1 for s, e in seq[\"loss_spans\"] for j in range(max(s, 1), min(e, len(ids)))})\n        x = torch.tensor([ids], device=dev)\n        hs = inner(input_ids=x, attention_mask=torch.ones_like(x), use_cache=False).last_hidden_state[0]\n        hd = lm_head.weight.device\n        h_sel = hs[torch.tensor(pos, device=hs.device)].to(hd)\n        t_ids = torch.tensor(seq[\"t_ids\"], dtype=torch.long, device=hd)\n        t_lp = torch.tensor(seq[\"t_lp\"], dtype=torch.float32, device=hd)\n        total = None\n        for a in range(0, len(pos), CH):\n            part = checkpoint(chunk_loss, h_sel[a:a + CH], t_ids[a:a + CH], t_lp[a:a + CH], use_reentrant=False)\n            total = part if total is None else total + part\n        (total / denom).backward()\n        return float(total.detach()), len(pos)\n\n    def cleanup() -> None:\n        opt.zero_grad(set_to_none=True)\n        gc.collect()\n        torch.cuda.empty_cache()\n        gc.collect()\n        torch.cuda.empty_cache()\n\n    def lr_at(step: int, total: int) -> float:       # linear warmup, then cosine decay to 10% of the peak\n        if step < WARMUP:\n            return LR * (step + 1) / WARMUP\n        prog = (step - WARMUP) / max(1, total - WARMUP)\n        return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * min(1.0, prog))))\n\n    order = list(range(len(seqs)))\n    groups = []\n    for ep in range(EPOCHS):\n        random.Random(SEED + ep).shuffle(order)\n        groups += [order[i:i + ACC] for i in range(0, len(order), ACC)]\n    total_steps = len(groups)\n    res.update({\"stage\": \"training\", \"optimizer_steps_planned\": total_steps, \"steps\": [], \"oom_events\": []})\n    for i in range(torch.cuda.device_count()):\n        torch.cuda.reset_peak_memory_stats(i)\n    t_train0 = time.perf_counter()\n    tok_done = 0\n    for step, grp in enumerate(groups):\n        lr = lr_at(step, total_steps)\n        for g in opt.param_groups:\n            g[\"lr\"] = lr\n        denom = float(sum(seqs[i][\"n_loss_tokens\"] for i in grp))\n        loss_sum, n_loss, n_tok, t1 = 0.0, 0, 0, time.perf_counter()\n        failed = False\n        for i in grp:\n            oom = False\n            try:\n                ls, nl = fwd_bwd(seqs[i], denom)\n                loss_sum += ls\n                n_loss += nl\n                n_tok += len(seqs[i][\"input_ids\"])\n            except torch.cuda.OutOfMemoryError:\n                oom = True\n            if oom:     # handled outside the except block so the failed step's tensors can be freed\n                res[\"oom_events\"].append({\"step\": step, \"task\": seqs[i][\"task\"], \"tokens\": len(seqs[i][\"input_ids\"])})\n                log(\"OOM on\", seqs[i][\"task\"], len(seqs[i][\"input_ids\"]), \"tokens; skipping this group\")\n                failed = True\n                break\n        if failed:\n            cleanup()\n            save()\n            continue\n        gn = float(torch.nn.utils.clip_grad_norm_(params, 1.0))\n        opt.step()\n        opt.zero_grad(set_to_none=True)\n        torch.cuda.synchronize()\n        dt = time.perf_counter() - t1\n        tok_done += n_tok\n        rec = {\"step\": step, \"lr\": round(lr, 7), \"loss\": round(loss_sum / max(1, n_loss), 4), \"grad_norm\": round(gn, 4), \"tokens\": n_tok, \"loss_tokens\": n_loss,\n               \"seconds\": round(dt, 1), \"tasks\": [seqs[i][\"task\"] for i in grp]}\n        res[\"steps\"].append(rec)\n        res[\"tokens_per_second\"] = round(tok_done / (time.perf_counter() - t_train0), 1)\n        res[\"peak_gib_per_gpu\"] = gpu_peaks()\n        save()\n        log(f\"step {step + 1}/{total_steps} lr {lr:.2e} loss {rec['loss']} gnorm {rec['grad_norm']} tokens {n_tok} {dt:.0f}s | {res['tokens_per_second']} tok/s | peak {gpu_peaks()}\")\n        if (step + 1) % CKPT_EVERY == 0:\n            pm.save_pretrained(str(OUT / \"adapter_ckpt\"))\n            log(\"checkpoint saved\")\n    leak = max((torch.cuda.memory_allocated(i) - b) for i, b in enumerate(base_alloc)) / 2**30 if base_alloc else 0.0\n    res[\"leak_gib_at_end\"] = round(leak, 2)\n    res[\"stage\"] = \"saving\"\n    pm.save_pretrained(str(OUT / \"adapter_full\"))\n    res[\"stage\"] = \"done\"\n    res[\"train_seconds\"] = round(time.perf_counter() - t_train0, 1)\n    save()\n    log(\"training finished: steps\", len(res[\"steps\"]), \"of\", total_steps, \"| tok/s\", res.get(\"tokens_per_second\"), \"| adapter saved\")\n\n\nif __name__ == \"__main__\":\n    try:\n        main()\n    except BaseException as e:  # noqa: BLE001\n        res[\"stage\"] = \"FAILED\"\n        res[\"error\"] = f\"{type(e).__name__}: {str(e)[:500]}\"\n        res[\"traceback\"] = traceback.format_exc()[-3000:]\n        save()\n        traceback.print_exc()\n        sys.exit(1)\n", "opd_loss.py": "\"\"\"Per-token distillation loss for on-policy distillation (OPD) from teacher top-k log-probabilities.\n\nThe teacher's top-k log-probs (from vLLM prompt_logprobs) are TRUE log-probs over the full vocabulary, so the probability of \"any other token\" is known too:\np_rest = 1 - sum_k p(x_k). The loss is the exact KL divergence between the student and teacher distributions coarse-grained to k+1 categories (the teacher's\ntop-k tokens, plus one bucket for the rest of the vocabulary). By the data-processing inequality it is a lower bound on the full-vocabulary KL, and it is 0 iff\nthe student puts the same mass as the teacher on each top-k token and on the remainder. No hyperparameter, no renormalisation that would hide student mass\noutside the teacher's support.\n\n  reverse KL (default, mode-seeking, MiniLLM / on-policy distillation):   KL(q || p) = sum_c q_c (log q_c - log p_c)\n  forward KL:                                                            KL(p || q) = sum_c p_c (log p_c - log q_c)\n\nInputs per position: student logits [n, V] (after softcapping), teacher top-k ids [n, k] (int64; -1 = padding) and log-probs [n, k] (float).\n\"\"\"\n\nimport torch\n\nMIN_P = 1e-6\n\n\ndef _teacher(t_ids, t_lp):\n    valid = t_ids >= 0\n    lp = torch.where(valid, t_lp.float(), torch.full_like(t_lp.float(), -float(\"inf\")))\n    p_top = lp.exp()                                                     # [n, k], 0 at padding\n    p_rest = (1.0 - p_top.sum(-1)).clamp_min(MIN_P)                      # [n]\n    return valid, p_top, p_rest\n\n\ndef coarse_kl(logits, t_ids, t_lp, mode=\"rkl\"):\n    \"\"\"Returns the per-position loss [n] (float32).\"\"\"\n    logq = torch.log_softmax(logits.float(), dim=-1)                     # [n, V]\n    valid, p_top, p_rest = _teacher(t_ids, t_lp)\n    lq_top = torch.gather(logq, 1, t_ids.clamp_min(0))                   # [n, k]\n    lq_top = torch.where(valid, lq_top, torch.full_like(lq_top, -float(\"inf\")))\n    q_top = lq_top.exp()\n    lq_rest = torch.log((1.0 - q_top.sum(-1)).clamp_min(MIN_P))          # [n]\n    lp_top = torch.where(valid, t_lp.float(), torch.zeros_like(lq_top))  # finite at padding (masked below)\n    lp_rest = p_rest.log()\n    if mode == \"rkl\":\n        top = torch.where(valid, q_top * (lq_top.clamp_min(-1e4) - lp_top), torch.zeros_like(q_top)).sum(-1)\n        rest = lq_rest.exp() * (lq_rest - lp_rest)\n    elif mode == \"fkl\":\n        top = torch.where(valid, p_top * (lp_top - lq_top.clamp_min(-1e4)), torch.zeros_like(p_top)).sum(-1)\n        rest = p_rest * (lp_rest - lq_rest)\n    else:\n        raise ValueError(mode)\n    return top + rest\n"}.items(): (CODE / n).write_text(txt, encoding='utf-8')   # OPD trainer + loss
sys.path.insert(0, str(CODE))
DATA_DIR = next(p.parent for p in Path('/kaggle/input').rglob('tasks.jsonl') if (p.parent / 'sample_submission').is_dir() and (p.parent / 'snapshots').is_dir()); TASKS_PATH = DATA_DIR / 'tasks.jsonl'
_sft = sorted(Path('/kaggle/input').rglob('opd_sequences.jsonl')); assert len(_sft) == 1, _sft; SFT = _sft[0]
_arms = [p.parent for p in Path('/kaggle/input').rglob('agent.yaml') if 'swe_scaffold_v5_e4b_base' in p.read_text(encoding='utf-8') and 'adapter:' not in p.read_text(encoding='utf-8')]
assert len(_arms) == 1, f'expected exactly one E4B arm, found {_arms}'
_b1e = WORK / 'e4b_arm'; shutil.rmtree(_b1e, ignore_errors=True); shutil.copytree(_arms[0], _b1e); B1 = _b1e
_sp = B1 / 'configs' / 'sampling.yaml'; _s = _sp.read_text(encoding='utf-8')   # G22: cap the reasoning (the only change to the arm)
assert _s.count('thinking_budget: 4096') == 1 and _s.count('max_output_tokens: 8192') == 1, _s
_sp.write_text(_s.replace('thinking_budget: 4096', 'thinking_budget: 512').replace('max_output_tokens: 8192', 'max_output_tokens: 4096'), encoding='utf-8')
print('sampling.yaml (capped):', _sp.read_text(encoding='utf-8')[-160:])
_ec = (B1 / 'eval_config.yaml').read_text(encoding='utf-8'); assert 'max_tool_calls: 28' in _ec and 'max_time_minutes: 8' in _ec, _ec   # the B2 regime
assert (B1 / 'agent.yaml').read_text(encoding='utf-8').count('gemma-4-e4b-it-qat-w4a16-ct') >= 1
import torch
_gpus = [(torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(torch.cuda.device_count())]; print('GPUs:', _gpus)
assert len(_gpus) == 4 and all(c[0] >= 8 for _, c in _gpus), f'expected 4x L4, got {_gpus}'
bf = sorted(p.parent for p in Path('/kaggle/input').rglob('config.json') if 'unquantized' in str(p).lower()); assert len(bf) == 1, bf
MODEL_BF16 = bf[0]
ALL_IDS = ["fastapi_11194", "fastapi_13537", "fastapi_13920", "fastapi_14077", "fastapi_14258", "fastapi_14301", "fastapi_14306", "fastapi_14372", "fastapi_14419", "fastapi_14448", "fastapi_14458", "fastapi_14463", "fastapi_14479", "fastapi_14487", "fastapi_14492", "fastapi_14616", "fastapi_14786", "fastapi_14794", "fastapi_14851", "fastapi_14873", "fastapi_14986", "fastapi_15280", "fastapi_15588", "fastapi_15589", "fastapi_15661", "fastapi_15800", "fastapi_5077", "fastapi_5624", "fastapi_9425", "fastapi_9555", "fastapi_9753", "requests_6644", "rich_2725", "rich_2943", "rich_3006", "rich_3043", "rich_3052", "rich_3061", "rich_3063", "rich_3067", "rich_3105", "rich_3180", "rich_3278", "rich_3454", "rich_3468", "rich_3470", "rich_3471", "rich_3472", "rich_3480", "rich_3506", "rich_3518", "rich_3521", "rich_3535", "rich_3675", "rich_3676", "rich_3718", "rich_3772", "rich_3777", "rich_3782", "rich_3882", "rich_3894", "rich_3905", "rich_3930", "rich_3934", "rich_3935", "rich_3938", "rich_3942", "rich_3944", "rich_3953", "rich_4006", "rich_4070", "rich_4075", "rich_4076", "rich_4077", "rich_4079"]
TRAINED = sorted({json.loads(l)['task'] for l in open(SFT, encoding='utf-8')}); UNSEEN = [i for i in ALL_IDS if i not in TRAINED]
print('sft:', SFT, '| B1 arm:', B1, '| bf16:', MODEL_BF16, '| trained tasks:', len(TRAINED), '| unseen tasks:', len(UNSEEN))
R['trained_tasks'] = TRAINED; save_results()
from swegemma.models import load_tasks
by_id = {t.instance_id: t for t in load_tasks(TASKS_PATH)}
assert not [i for i in ALL_IDS if i not in by_id]
SMOKE_IDS = random.Random(5).sample([i for i in ALL_IDS if i.startswith(('fastapi_', 'rich_'))], 4)

## Fail-fast: adapter wiring through the harness compile path

In [ ]:
import g3_wiring as W
# Fail fast, before ~3 h of training: does an adapter flow through the harness's own compile path? (placeholder = the organizers' sample adapter)
t0 = time.time()
arm0 = W.make_eval_arm(B1, DATA_DIR / 'sample_submission' / 'adapters' / 'main_lora', Path('/tmp/wiring_arm'))
R['wiring_placeholder'] = W.check_wiring(arm0); print('WIRING OK with placeholder adapter:', R['wiring_placeholder'], round(time.time() - t0, 1), 's'); save_results()

## Load the G21 adapter (no training)

In [ ]:
import hashlib
t0 = time.time()
_ad = sorted(p.parent for p in Path('/kaggle/input').rglob('adapter_model.safetensors') if 'g22-adapter' in str(p) and (p.parent / 'adapter_config.json').exists()); assert len(_ad) == 1, _ad
assert hashlib.sha256((_ad[0] / 'adapter_model.safetensors').read_bytes()).hexdigest() == '348f020e3ab6f35f03f5edc267ac44e47a75730cb6d2c85a0165c1a3d28396b8', 'not the G21 adapter'
shutil.copytree(_ad[0], G3 / 'train' / 'adapter_full', dirs_exist_ok=True)
R['train'] = {'source': 'G21 adapter, kernel noobhunterjay/gemma-4-g21-e4b-opd v1 (122 steps coarse top-8 reverse KL)', 'path': str(_ad[0]),
              'sha256': hashlib.sha256((_ad[0] / 'adapter_model.safetensors').read_bytes()).hexdigest()}
R['train_wall_s'] = round(time.time() - t0); save_results(); print(R['train'])


## Package adapter, re-check wiring with the trained adapter

In [ ]:
os.environ['SCORER_MODEL'] = 'google/gemma-4-e4b-it-qat-w4a16-ct'
import make_adapters
assert make_adapters.SCORER_MODEL == 'google/gemma-4-e4b-it-qat-w4a16-ct', make_adapters.SCORER_MODEL
rep = make_adapters.main(G3 / 'train' / 'adapter_full', G3 / 'adapters', DATA_DIR / 'sample_submission' / 'adapters' / 'main_lora')
R['adapter'] = {'bytes': rep['g1_full']['bytes'], 'tensors': rep['g1_full']['tensors'], 'target_modules': rep['g1_full']['target_modules']}
from safetensors.torch import load_file
tt = load_file(str(G3 / 'adapters' / 'g1_full' / 'adapter_model.safetensors'))
bs = [v.float().abs().mean().item() for k, v in tt.items() if 'lora_B' in k]
R['adapter']['mean_abs_lora_B'] = sum(bs) / len(bs); R['adapter']['nonzero_B_fraction'] = sum(1 for x in bs if x > 0) / len(bs)
EVAL_ARM = W.make_eval_arm(B1, G3 / 'adapters' / 'g1_full', WORK / 'eval_arm')
_es = (EVAL_ARM / 'configs' / 'sampling.yaml').read_text(encoding='utf-8'); assert 'thinking_budget: 512' in _es and 'max_output_tokens: 4096' in _es, _es
R['wiring_real'] = W.check_wiring(EVAL_ARM); print('adapter:', R['adapter']); print('WIRING OK with the trained adapter:', R['wiring_real']); save_results()
assert R['adapter']['mean_abs_lora_B'] > 0, 'trained adapter has all-zero B matrices'
(WORK / 'eval_arm' / 'adapters' / 'g3_lora' / 'adapter_config.json').read_text()

## vLLM with the adapter + capture proxy

In [ ]:
import litellm, torch
from adk_submission import VllmConfig, VllmServer, discover_adapters
from swegemma.config import ALLOWED_ADAPTER_EXTENSIONS
litellm.drop_params = True
TARGET_MODEL_NAME = 'gemma-4-e4b-it-qat-w4a16-ct'
MODEL_PATH = Path('/kaggle/input/models/google/gemma-4/other/gemma-4-e4b-it-qat-w4a16-ct/2')
first_arm = EVAL_ARM
try:   # swegemma >= 0.2.11 / adk_submission >= 0.2.13 (wheelhouse of 2026-10-09)
    from adk_submission import discover_declared_models
    _declared = discover_declared_models(first_arm); assert len(_declared) == 1, f'expected exactly one declared model, got {_declared}'
    declared_model = next(iter(_declared))
except ImportError:   # older wheelhouse
    from swegemma.models.discovery import validate_single_declared_model
    declared_model = validate_single_declared_model(first_arm)
adapters = discover_adapters(str(first_arm), adapter_extensions=ALLOWED_ADAPTER_EXTENSIONS)
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 1
tp_size = 4 if gpu_count >= 4 else (2 if gpu_count >= 2 else 1)
vllm_cfg = VllmConfig(model=str(MODEL_PATH), port=8000, host='127.0.0.1', tool_call_parser='gemma4', reasoning_parser='gemma4',
    max_model_len=32768, dtype='bfloat16' if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else 'auto',
    gpu_memory_utilization=0.90, enable_auto_tool_choice=True, enable_lora=True, max_loras=8, max_lora_rank=128,
    tensor_parallel_size=tp_size, startup_timeout=60 * 12)
server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)
server_instance.start()
print(f'vLLM server started on {server_instance.base_url} (tp={tp_size})')
import capture_proxy as CP
VLLM_URL = f'http://127.0.0.1:{server_instance.config.port}'
proxy = CP.CaptureProxy(VLLM_URL, str(G3 / 'capture.jsonl.gz'), port=8001).start()
server_instance.config.port = 8001      # the harness now talks to the capture proxy
models = server_instance.create_model_registry(aliases=[declared_model, TARGET_MODEL_NAME], model_prefix='openai/', api_key='EMPTY')
print('registry aliases:', sorted(getattr(models, '_registry', {}).keys()) if hasattr(models, '_registry') else 'n/a')
print('harness endpoint (proxy):', server_instance.base_url, '-> upstream', VLLM_URL, '| server cmd has --enable-lora:', '--enable-lora' in ' '.join(server_instance.build_cmd()))

## Smoke gate and evaluation round

In [ ]:
import asyncio, concurrent.futures, yaml
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps._configs import EventsCompactionConfig
from swegemma.config import EvalConfig, build_submission_limits
from swegemma.evaluate import Evaluator
def run_sync(coro_fn):
    try: loop = asyncio.get_running_loop()
    except RuntimeError: loop = None
    if loop is not None and loop.is_running():
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool: return pool.submit(lambda: asyncio.run(coro_fn())).result()
    return asyncio.run(coro_fn())
def budget_of(d):
    cfg = yaml.safe_load((d / 'eval_config.yaml').read_text(encoding='utf-8')).get('evaluation', {}); out = {}
    for k, cast in (('timeout_seconds', int), ('max_tool_calls', int), ('max_time_minutes', float)):
        if k in cfg: out[k] = cast(cfg[k])
    t = cfg.get('max_turns', cfg.get('max_llm_calls'))
    if t is not None: out['max_turns'] = int(t)
    return out
limits, gen_constraints = build_submission_limits()
def run_round(tag, ids, arm_dir=None):
    arm_dir = arm_dir or EVAL_ARM
    proxy.round_tag = tag; rd = G3 / tag
    cfg = EvalConfig(tasks_path=TASKS_PATH, snapshots_dir=DATA_DIR / 'snapshots', results_dir=rd, submission_dir=arm_dir, models=models, sandbox='subprocess',
        task_ids=list(ids), concurrency=4, display_mode='quiet', limits=limits, generation_constraints=gen_constraints, adapter_manifest=adapters,
        context_cache_config=ContextCacheConfig(min_tokens=2048, ttl_seconds=1800, cache_intervals=10),
        events_compaction_config=EventsCompactionConfig(compaction_interval=15, overlap_size=2, token_threshold=14336, event_retention_size=5),
        graph_dir=str(DATA_DIR / 'graphs'), embeddings_dir=str(DATA_DIR / 'embeddings'), wheels_dir=DATA_DIR / 'wheels', verbose=False, **budget_of(arm_dir))
    t0 = time.perf_counter()
    try: run_sync(Evaluator(cfg).run)
    except Exception:
        traceback.print_exc(); R[tag] = {'error': 'run raised', 'trace': traceback.format_exc()[-1500:]}; save_results(); return None
    rows = [json.loads(l) for l in open(rd / 'task_results.jsonl', encoding='utf-8')]
    ok = {r['instance_id'] for r in rows if r['resolved']}
    R[tag] = {'n': len(rows), 'resolved': len(ok), 'patch_written': sum(1 for r in rows if r['agent_patch_size'] > 0),
              'time_cap_hit': sum(1 for r in rows if 'session timeout' in (r['error'] or '')), 'wall_s': round(time.perf_counter() - t0, 1),
              'mean_task_s': round(sum(r['duration_seconds'] for r in rows) / max(1, len(rows)), 1), 'resolved_ids': sorted(ok),
              'resolved_on_trained_tasks': len(ok & set(TRAINED)), 'resolved_on_unseen_tasks': len(ok & set(UNSEEN)), 'proxy': proxy.stats()}
    save_results(); print(tag, {k: v for k, v in R[tag].items() if k != 'resolved_ids'}, flush=True); return rows

# ---- adapter-effect gate: the same temperature-0 request to the base and to the adapter must differ (a silently inactive LoRA would pass the name check below)
import requests
def _ask(model, prompt):
    body = {'model': model, 'messages': [{'role': 'user', 'content': prompt}], 'temperature': 0, 'max_tokens': 64, 'logprobs': True, 'top_logprobs': 1, 'chat_template_kwargs': {'enable_thinking': False}}
    r = requests.post(VLLM_URL + '/v1/chat/completions', json=body, timeout=600); r.raise_for_status(); ch = r.json()['choices'][0]
    return ch['message'].get('content') or '', [t['logprob'] for t in (ch.get('logprobs') or {}).get('content', [])]
_eff = []
for _p in ('Write a Python function that returns the n-th Fibonacci number, then explain it in one sentence.', 'The bug: rich.table.Table drops the caption when show_header=False. Which file would you look at first, and why?'):
    _tb, _lb = _ask(str(MODEL_PATH), _p); _ta, _la = _ask('g3_lora', _p); _n = min(len(_lb), len(_la))
    _eff.append({'text_differs': _tb != _ta, 'mean_abs_logprob_diff': round(sum(abs(a - b) for a, b in zip(_lb[:_n], _la[:_n])) / max(_n, 1), 4)})
R['adapter_effect'] = _eff; save_results(); print('adapter effect (base vs adapter, temperature 0):', _eff)
if not any(e['text_differs'] or e['mean_abs_logprob_diff'] > 0.01 for e in _eff): raise RuntimeError(f'ADAPTER HAS NO EFFECT on the E4B in vLLM: {_eff}')

# ---- smoke round + gate: requests must really go to the adapter (model name g3_lora), capture must work
rows = run_round('smoke', SMOKE_IDS)
st = proxy.stats(); recs = list(CP.read_records(str(G3 / 'capture.jsonl.gz')))
main = [r for r in recs if (next((m['content'] for m in r['request']['messages'] if m['role'] == 'system'), '') or '').startswith('You are an expert autonomous')]
used = sum(1 for r in main if r['request'].get('model') == 'g3_lora')
gate = {'rows': bool(rows) and len(rows) == len(SMOKE_IDS), 'records >= 12': st['records'] >= 12, 'no proxy errors': st['errors'] == 0,
        'main-agent requests use the adapter (>= 95%)': bool(main) and used / len(main) >= 0.95,
        'requests carry thinking_token_budget 512': bool(main) and all(r['request'].get('thinking_token_budget') == 512 for r in main),
        'requests carry max_completion_tokens 4096': bool(main) and all(r['request'].get('max_completion_tokens', r['request'].get('max_tokens')) == 4096 for r in main)}
R['smoke_gate'] = gate; R['smoke_model_names'] = sorted({r['request'].get('model') for r in recs}); save_results(); print('smoke gate:', gate, '| model names seen:', R['smoke_model_names'], f'| adapter used in {used}/{len(main)} main requests')
if not all(gate.values()): raise RuntimeError(f'SMOKE GATE FAILED, stopping before the evaluation round: {gate} {st}')

# ---- evaluation round over all 75 tasks with the adapter
ids = list(ALL_IDS); random.Random(201).shuffle(ids)
run_round('e1', ids)
rep = R.get('e1', {})
print('adapter @512 round e1:', {k: v for k, v in rep.items() if k not in ('resolved_ids', 'proxy')}, flush=True)
# ---- control: base E4B @512 on the same server, same task order (the watchdog may stop it; e1 is already saved)
run_round('b512', ids, arm_dir=B1)
_b = [r for r in CP.read_records(str(G3 / 'capture.jsonl.gz')) if r.get('round') == 'b512']
R['b512_model_names'] = sorted({r['request'].get('model') for r in _b}); save_results(); print('b512 model names:', R['b512_model_names'])
print('\n===== G22 SUMMARY (G21 adapter, reasoning capped at 512) =====')
print('adapter:', R['train'])
print('adapter @512 e1:', {k: v for k, v in rep.items() if k not in ('resolved_ids', 'proxy')})
print('base @512 b512:', {k: v for k, v in R.get('b512', {}).items() if k not in ('resolved_ids', 'proxy')})
print('BASE E4B (same arm, no adapter; G14c e4b_b2): 7/75 resolved, ~69 patches, <= 1 capped, ~120 s per task; other base rounds 10 and 8; the 31B base solves 25 and 28')
print('G21 (adapter @4096, 60 tasks): 6 resolved, 9 patches, 52 capped')
print('RULE (pre-registered): SIGNAL adapter >= 12 & capped <= 8 & >= base@512 + 3 -> masked OPD retrain | NULL <= 11 or < base@512 + 3 | FAILURE capped > 20')
R['finished'] = time.strftime('%Y-%m-%d %H:%M:%S'); R['capture'] = proxy.stats(); save_results(); _wd.cancel()